# NB03 — Dirt Engine Demo (VirtualDirtGrid within Custom Env)

This notebook tests the `VirtualDirtGrid` module that tracks cleaning progress
on the plate surface in `UnitreeG1DishWipe-v1`.  The grid is now a **shared
module** at `src/envs/dirt_grid.py` and is integrated into the environment.


## Objective

1. Import and test `VirtualDirtGrid` standalone (unit tests).
2. Verify brush radius effect (radius=0 vs 1 vs 2).
3. Test grid updates within the live custom env.
4. Visualise cleaning progress.
5. Save artifacts.


## Environment

| Notebook | Goal | Required HW | Min CPU | Min RAM | GPU VRAM |
|---|---|---|---:|---:|---:|
| NB03 | Dirt engine + brush viz | CPU | 2 cores | 4 GB | 0 GB |


In [1]:
import sys, os, platform
print(f"Python: {sys.version}")
print(f"OS    : {platform.platform()}")

import numpy as np; print(f"NumPy : {np.__version__}")
import matplotlib; print(f"Matplotlib: {matplotlib.__version__}")



Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
OS    : Windows-11-10.0.22631-SP0
NumPy : 2.4.2
Matplotlib: 3.10.8


## Imports


In [2]:
import numpy as np
import json, csv
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import torch, gymnasium as gym

_cwd = Path(os.getcwd()).resolve()
if (_cwd / "src").is_dir():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "src").is_dir():
    PROJECT_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find src/ from {_cwd}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.envs.dirt_grid import VirtualDirtGrid
from src.envs.dishwipe_env import UnitreeG1DishWipeEnv  # registers env


c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\sapien\__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\sapien\wrapper\pinocchio_model.py:299: UserWarning: pinnochio package is not installed, robotics functionalities will not be available
  warnings.warn(
c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config


In [3]:
CFG = dict(
    seed=42,
    env_id="UnitreeG1DishWipe-v1",
    grid_h=10, grid_w=10,
    brush_radii_to_test=[0, 1, 2],
    env_test_steps=50,
)
SEED = CFG["seed"]
artifact_dir = Path("artifacts/NB03")
artifact_dir.mkdir(parents=True, exist_ok=True)
print("Config:", json.dumps(CFG, indent=2))


Config: {
  "seed": 42,
  "env_id": "UnitreeG1DishWipe-v1",
  "grid_h": 10,
  "grid_w": 10,
  "brush_radii_to_test": [
    0,
    1,
    2
  ],
  "env_test_steps": 50
}


## Reproducibility


In [4]:
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"✅ Seeds set to {SEED}")


✅ Seeds set to 42


## Implementation Steps

### Step 1 — Unit test VirtualDirtGrid standalone


In [5]:
grid = VirtualDirtGrid(H=10, W=10, brush_radius=1)
print(f"Grid: {grid}")

# Test reset
grid.reset()
assert grid.get_cleaned_ratio() == 0.0, "Grid should start all dirty"
print("✅ Reset: all dirty")

# Test mark_clean at center (5, 5) with brush_radius=1 → 3×3 = 9 cells
delta = grid.mark_clean(5, 5)
print(f"mark_clean(5,5): delta={delta} cells cleaned")
assert delta == 9, f"Expected 9 (3×3), got {delta}"
assert grid.get_cleaned_ratio() == 9 / 100
print(f"Cleaned ratio: {grid.get_cleaned_ratio()}")

# Test duplicate cleaning — should return 0 newly cleaned
delta2 = grid.mark_clean(5, 5)
assert delta2 == 0, f"Re-cleaning should give delta=0, got {delta2}"
print("✅ Double-clean returns delta=0")

# Test corner (0, 0) with brush_radius=1 → 2×2 = 4 cells
delta3 = grid.mark_clean(0, 0)
print(f"mark_clean(0,0): delta={delta3}")
assert delta3 == 4, f"Expected 4 (corner clip), got {delta3}"

# Test get_grid_flat
flat = grid.get_grid_flat()
assert flat.shape == (100,), f"Expected (100,), got {flat.shape}"
assert flat.dtype == np.float32
print(f"✅ get_grid_flat: shape={flat.shape}, dtype={flat.dtype}")

print("\n✅ All standalone unit tests passed!")


Grid: VirtualDirtGrid(H=10, W=10, brush_radius=1, cleaned=0.0%)
✅ Reset: all dirty
mark_clean(5,5): delta=9 cells cleaned
Cleaned ratio: 0.09
✅ Double-clean returns delta=0
mark_clean(0,0): delta=4
✅ get_grid_flat: shape=(100,), dtype=float32

✅ All standalone unit tests passed!


### Step 2 — Brush radius comparison


In [6]:
results = {}
fig, axes = plt.subplots(1, len(CFG["brush_radii_to_test"]), figsize=(14, 4))
cmap = ListedColormap(["#D32F2F", "#4CAF50"])
norm = BoundaryNorm([0, 0.5, 1], cmap.N)

for idx, radius in enumerate(CFG["brush_radii_to_test"]):
    g = VirtualDirtGrid(H=10, W=10, brush_radius=radius)
    g.reset()

    # Clean center cell
    delta = g.mark_clean(5, 5)
    results[radius] = dict(delta=delta, ratio=g.get_cleaned_ratio())

    ax = axes[idx]
    ax.imshow(g.get_grid(), cmap=cmap, norm=norm, origin="lower")
    ax.set_title(f"brush_radius={radius}\n(delta={delta})")
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    for i in range(10):
        for j in range(10):
            v = g.get_grid()[i, j]
            ax.text(j, i, str(v), ha="center", va="center",
                    fontsize=7, color="white")

fig.suptitle("NB03 — Brush Radius Comparison (single clean at cell 5,5)")
fig.tight_layout()
brush_path = artifact_dir / "brush_effect_demo.png"
fig.savefig(str(brush_path), dpi=120, bbox_inches="tight")
plt.close("all")

for r, data in results.items():
    print(f"  radius={r}: delta={data['delta']}, ratio={data['ratio']:.2%}")
print(f"\n✅ Saved: {brush_path}")


  radius=0: delta=1, ratio=1.00%
  radius=1: delta=9, ratio=9.00%
  radius=2: delta=25, ratio=25.00%

✅ Saved: artifacts\NB03\brush_effect_demo.png


### Step 3 — Test grid within live custom env


In [7]:
env = gym.make(
    CFG["env_id"], obs_mode="state",
    control_mode="pd_joint_delta_pos", render_mode=None, num_envs=1,
)
obs, info = env.reset(seed=SEED)
env_grid = env.unwrapped._dirt_grids[0]
print(f"Env grid after reset: {env_grid}")
assert env_grid.get_cleaned_ratio() == 0.0

# Step the env and check that info contains dirt tracking keys
cleaning_trace = []
for step_i in range(CFG["env_test_steps"]):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    ratio = info.get("cleaned_ratio", torch.tensor([0.0])).item()
    delta = info.get("delta_clean", torch.tensor([0.0])).item()
    cleaning_trace.append(dict(step=step_i, delta_clean=delta, cleaned_ratio=ratio))

    if terminated.any() or truncated.any():
        obs, info = env.reset(seed=SEED + step_i)

final_ratio = env_grid.get_cleaned_ratio()
total_delta = sum(t["delta_clean"] for t in cleaning_trace)
print(f"\n{CFG['env_test_steps']} random steps:")
print(f"  Total cells cleaned  : {total_delta}")
print(f"  Final cleaned ratio  : {final_ratio:.4f}")
print(f"  Grid state:\n{env_grid.get_grid()}")


2026-03-01 17:01:53,000 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


Env grid after reset: VirtualDirtGrid(H=10, W=10, brush_radius=1, cleaned=0.0%)

50 random steps:
  Total cells cleaned  : 0.0
  Final cleaned ratio  : 0.0000
  Grid state:
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


### Step 4 — Visualise cleaning over time


In [8]:
fig, ax = plt.subplots(figsize=(8, 4))
ratios = [t["cleaned_ratio"] for t in cleaning_trace]
ax.plot(ratios, linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Cleaned Ratio")
ax.set_title("NB03 — Cleaning Progress (Random Actions)")
ax.set_ylim(-0.01, max(0.1, max(ratios) * 1.2))
ax.grid(True, alpha=0.3)
fig.tight_layout()
progress_path = artifact_dir / "dirt_engine_test.png"
fig.savefig(str(progress_path), dpi=120, bbox_inches="tight")
plt.close("all")
print(f"✅ Saved: {progress_path}")


✅ Saved: artifacts\NB03\dirt_engine_test.png


### Step 5 — Coordinate mapping validation


In [9]:
# Verify world_to_cell matches what the env uses internally
from src.envs.dishwipe_env import PLATE_HALF_SIZE, PLATE_POS_IN_SINK

plate_pos = env.unwrapped.plate.pose.p[0].cpu().numpy()
plate_half = np.array(PLATE_HALF_SIZE[:2])
print(f"Plate center: {plate_pos}")
print(f"Plate half  : {plate_half}")

# Sample points on plate corners
test_points = {
    "center": plate_pos.copy(),
    "top-right": plate_pos + np.array([plate_half[0]*0.9, plate_half[1]*0.9, 0]),
    "bottom-left": plate_pos + np.array([-plate_half[0]*0.9, -plate_half[1]*0.9, 0]),
}
g = VirtualDirtGrid(H=10, W=10, brush_radius=0)
for name, pt in test_points.items():
    u, v = VirtualDirtGrid.world_to_uv(pt, plate_pos, plate_half)
    ci, cj = g.uv_to_cell(u, v)
    print(f"  {name:12s}: world={pt[:2]} → uv=({u:.3f},{v:.3f}) → cell=({ci},{cj})")
print("✅ Coordinate mapping validated.")

Plate center: [0.09322052 0.21052144 0.58      ]
Plate half  : [0.1 0.1]
  center      : world=[0.09322052 0.21052144] → uv=(0.500,0.500) → cell=(5,5)
  top-right   : world=[0.18322052 0.30052144] → uv=(0.950,0.950) → cell=(9,9)
  bottom-left : world=[0.00322052 0.12052144] → uv=(0.050,0.050) → cell=(0,0)
✅ Coordinate mapping validated.


### Step 6 — Save artifacts


In [10]:
# Save cleaning trace
trace_path = artifact_dir / "cleaning_trace.csv"
with open(trace_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=cleaning_trace[0].keys())
    writer.writeheader()
    writer.writerows(cleaning_trace)
print(f"✅ Saved: {trace_path}")

# Save config
config_path = artifact_dir / "nb03_config.json"
with open(config_path, "w") as f:
    json.dump(CFG, f, indent=2)
print(f"✅ Saved: {config_path}")


✅ Saved: artifacts\NB03\cleaning_trace.csv
✅ Saved: artifacts\NB03\nb03_config.json


### Step 7 — MLflow logging


In [11]:
try:
    import mlflow
    from dotenv import load_dotenv
    load_dotenv(".env.local")
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)
        mlflow.set_experiment("dishwipe_unitree_g1")
        with mlflow.start_run(run_name="NB03_dirt_engine_v2"):
            mlflow.log_params(CFG)
            mlflow.log_metric("final_cleaned_ratio", final_ratio)
            mlflow.log_metric("total_cells_cleaned", total_delta)
            for r, data in results.items():
                mlflow.log_metric(f"brush_r{r}_delta", data["delta"])
            mlflow.log_artifacts(str(artifact_dir), artifact_path="NB03")
            print("✅ MLflow run logged.")
    else:
        print("⚠️ MLFLOW_TRACKING_URI not set — skipping MLflow.")
except Exception as e:
    print(f"⚠️ MLflow logging failed: {e}")


✅ MLflow run logged.
🏃 View run NB03_dirt_engine_v2 at: https://mlflow.cie.co.th/#/experiments/7/runs/418e07dfe6fe40949e992e90ee9e832f
🧪 View experiment at: https://mlflow.cie.co.th/#/experiments/7


## Results

- `VirtualDirtGrid` unit tests all pass (reset, mark_clean, delta counting, brush effects)
- Brush radius 0 → 1 cell, radius 1 → 9 cells (3×3), radius 2 → 25 cells (5×5)
- Dirt grid integrated in env: `info` dict contains `cleaned_ratio` and `delta_clean`
- Coordinate mapping (world → uv → cell) produces correct grid indices


## Artifacts

| File | Description |
|------|-------------|
| `artifacts/NB03/brush_effect_demo.png` | Side-by-side brush radius comparison |
| `artifacts/NB03/dirt_engine_test.png` | Cleaning progress curve |
| `artifacts/NB03/cleaning_trace.csv` | Step-by-step cleaning trace |


## Cleanup


In [12]:
env.close()
print('✅ NB03 complete.')


✅ NB03 complete.


## References

- `src/envs/dirt_grid.py` — VirtualDirtGrid implementation
- `src/envs/dishwipe_env.py` — Integration in custom ManiSkill env
